# Stable Audio 3 Small Music — Game Music Quality Pilot

**Purpose:** test whether a dedicated, commercially usable music checkpoint can produce a polished, usable background-music bed for a game scene.

This is a small inference experiment only: **no training, no fine-tuning, no video input, and no external generation API**. We keep one prompt fixed and generate three seeds so we can judge musical quality and variation without changing several things at once.

The outputs are 30-second instrumental candidates. This notebook is a music-layer experiment alongside the existing Kandinsky video-to-audio work; it does not replace that workflow or test synchronization.

## Before running: model access and commercial terms

1. Accept the access and license conditions on the [Stable Audio 3 Small Music model page](https://huggingface.co/stabilityai/stable-audio-3-small-music). The page also requires acceptance of the [Gemma Terms of Use](https://ai.google.dev/gemma/terms) for its text encoder.
2. Add your Hugging Face **read token** to Colab Secrets as `HF_TOKEN`, and enable notebook access. Do not paste the token into a cell or print it.
3. Stability AI's Community License permits qualifying commercial use below its $1M annual-revenue threshold; Stability says commercial product/service research must be registered. Review the current [license and registration requirements](https://stability.ai/license) before product integration.

The notebook records the model revision and generation settings. It does not establish legal clearance.

## Step 1 — choose a Colab GPU runtime

Use **Runtime → Change runtime type → A100** if available. The Small Music model is much lighter than the video model, but the first run still downloads the environment and gated weights. Subsequent generation cells reuse the loaded model.

In [ ]:
import os, shutil, subprocess, sys, time, json, textwrap
from pathlib import Path

print("Python:", sys.version.split()[0])
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    )
    print("GPU:", result.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU not detected. Select a GPU runtime in Colab before continuing.")

## Step 2 — install the official inference package in an isolated Python environment

The setup uses the official Stable Audio 3 repository and its lockfile inside a separate Python 3.10 environment. That keeps Colab's own notebook packages intact and avoids mixing PyTorch versions with the Kandinsky environment.

In [ ]:
REPO = Path("/content/stable-audio-3")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-audio-3.git", str(REPO)],
        check=True
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was installed but is not available on PATH. Restart the runtime and rerun this setup cell.")

subprocess.run([UV, "python", "install", "3.10"], check=True)
subprocess.run([UV, "sync", "--project", str(REPO), "--python", "3.10"], check=True)

PYTHON = REPO / ".venv" / "bin" / "python"
if not PYTHON.exists():
    raise FileNotFoundError(f"Isolated Python environment was not created: {PYTHON}")

REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Stable Audio 3 commit:", REVISION)
print("Environment ready:", PYTHON)

## Step 3 — configure one scene and keep it fixed

This baseline prompt describes an **instrumental background score**, not a sound effect. You can edit the prompt to match a scene you want to test. Keep it unchanged while comparing the three seeds. The prompt is in English because the model card lists English conditioning.

In [ ]:
PROMPT = (
    "Instrumental cinematic ambient game music for exploring an ancient forest at dawn; "
    "gentle flowing string motif, light harp, warm sustained pads, calm and curious mood, "
    "natural dynamics, restrained background score. No vocals, no lyrics, no percussion, no sound effects."
)
DURATION_SECONDS = 30
SEEDS = [42, 123, 7]
STEPS = 8

print("Candidates:", len(SEEDS), "| duration:", DURATION_SECONDS, "seconds")
print("Prompt:", PROMPT)

## Step 4 — load the Small Music checkpoint

The model is gated. The cell reads `HF_TOKEN` from Colab Secrets without displaying it. If access fails, confirm that you accepted the model page's conditions and enabled notebook access for the secret.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN
del HF_TOKEN

check = subprocess.run(
    [str(PYTHON), "-c", "import torch; print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'"],
    text=True, capture_output=True
)
print(check.stdout)
if check.returncode:
    print(check.stderr[-3000:])
    raise RuntimeError("GPU/PyTorch preflight failed.")

print("HF token is available to the model loader (value hidden).")

## Step 5 — generate three candidates

Only the seed changes between candidates. They run one at a time to keep peak memory low. Each WAV is saved at the model's native sample rate without extra normalization or post-processing.

In [ ]:
OUT_DIR = Path("/content/stable_audio_music_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)

generation_script = f"""
import json, time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from stable_audio_3 import StableAudioModel

prompt = {json.dumps(PROMPT)}
duration = {int(DURATION_SECONDS)}
seeds = {json.dumps(SEEDS)}
steps = {int(STEPS)}
out_dir = Path({json.dumps(str(OUT_DIR))})
out_dir.mkdir(parents=True, exist_ok=True)

model = StableAudioModel.from_pretrained("small-music", device="cuda", model_half=True)
sample_rate = int(model.model.sample_rate)
manifest = {{
    "schema": "stable-audio-3-small-music-pilot-v1",
    "model": "stabilityai/stable-audio-3-small-music",
    "repo_revision": {json.dumps(REVISION)},
    "prompt": prompt,
    "duration_seconds": duration,
    "steps": steps,
    "cfg_scale": 1.0,
    "seeds": seeds,
    "sample_rate_hz": sample_rate,
    "outputs": []
}}

for seed in seeds:
    started = time.time()
    audio = model.generate(
        prompt=prompt,
        duration=duration,
        steps=steps,
        cfg_scale=1.0,
        seed=int(seed),
        batch_size=1,
    )
    waveform = audio[0].detach().float().cpu().numpy().T
    filename = f"candidate_seed_{{seed}}.wav"
    output_path = out_dir / filename
    sf.write(output_path, waveform, sample_rate, subtype="PCM_16")
    elapsed = time.time() - started
    manifest["outputs"].append({{
        "file": filename,
        "seed": int(seed),
        "duration_seconds": round(len(waveform) / sample_rate, 3),
        "generation_seconds": round(elapsed, 2)
    }})
    print(f"Saved {{filename}} | {{len(waveform) / sample_rate:.2f}} s | {{elapsed:.1f}} s")
    del audio, waveform
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(out_dir / "pilot_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Sample rate:", sample_rate, "Hz")
"""

SCRIPT_PATH = Path("/content/run_stable_audio_music_pilot.py")
SCRIPT_PATH.write_text(generation_script, encoding="utf-8")
subprocess.run([str(PYTHON), str(SCRIPT_PATH)], check=True)

## Step 6 — listen and rate the candidates

Listen at a comfortable volume. Score each item from 1 (poor) to 5 (strong):

- **Musical quality:** does it sound intentional and coherent?
- **Scene fit:** does it suit the described game scene?
- **Background usability:** does it stay behind gameplay rather than demand attention?
- **Artifacts:** audible glitches, harshness, abrupt changes, or unwanted voice/percussion?
- **Variation:** do the seeds produce meaningfully different arrangements?

This pilot answers whether this model can produce a promising standalone music bed. It does **not** yet test video understanding, automatic prompt writing, looping, or synchronization.

In [ ]:
from IPython.display import Audio, display

for seed in SEEDS:
    wav_path = OUT_DIR / f"candidate_seed_{seed}.wav"
    print(f"Candidate — seed {seed}")
    display(Audio(filename=str(wav_path)))

## Step 7 — keep the base candidates ready

The base candidate WAVs and their manifest remain in `OUT_DIR`. The final download package is created after the inpainting comparison so it can include both the selected original and its edited version.

In [ ]:
print("Base candidates are ready in:", OUT_DIR)
print("Choose a preferred seed in the next step; the default is seed 42.")

## Step 8 — choose the base candidate and the instrument edit

Choose the seed of the candidate you liked best. The default is 42; change it to 123 or 7 if you preferred another candidate.

This test edits only seconds 12–20 of the chosen track and asks the model to introduce a soft harp part. It is **not a separate harp stem**: the result is a new full-mix version with the selected interval regenerated.

In [ ]:
BASE_SEED = 42  # Change to 123 or 7 to select another base candidate.
INPAINT_START_SECONDS = 12.0
INPAINT_END_SECONDS = 20.0
INPAINT_SEED = 314

INPAINT_PROMPT = (
    "Introduce a soft, delicate harp arpeggio underneath the existing instrumental "
    "arrangement, matching its tempo, harmony, mood, and texture. Keep the same "
    "musical identity. No vocals, lyrics, drums, or sound effects."
)

BASE_WAV = OUT_DIR / f"candidate_seed_{BASE_SEED}.wav"
INPAINT_WAV = OUT_DIR / f"inpainted_seed_{BASE_SEED}_harp_{int(INPAINT_START_SECONDS)}_{int(INPAINT_END_SECONDS)}.wav"

if not BASE_WAV.is_file():
    raise FileNotFoundError(
        f"Base candidate not found: {BASE_WAV}. Run the original generation cell first."
    )
if not (0 <= INPAINT_START_SECONDS < INPAINT_END_SECONDS <= DURATION_SECONDS):
    raise ValueError("The inpainting interval must lie inside the source track.")

print("Base candidate:", BASE_WAV.name)
print(f"Regenerated interval: {INPAINT_START_SECONDS:.1f}–{INPAINT_END_SECONDS:.1f} s")
print("Edit request:", INPAINT_PROMPT)
from IPython.display import Audio, display
display(Audio(filename=str(BASE_WAV)))

### What the next cell does

It loads the selected WAV as the reference, then calls Stable Audio 3's inpainting mode with a time mask. According to the official inference guide, the model uses the surrounding context and regenerates the masked interval while preserving the unmasked regions. In practice, listen for whether the harp is audible, whether the transition is smooth, and whether the rest of the piece stays intact. This is an inpainting edit, not guaranteed instrument separation. See the [official inference guide](https://github.com/Stability-AI/stable-audio-3/blob/main/docs/workflows/inference.md#inpaintingcontinuation).

In [ ]:
INPAINT_CONFIG = {
    "input_wav": str(BASE_WAV),
    "output_wav": str(INPAINT_WAV),
    "prompt": INPAINT_PROMPT,
    "duration_seconds": float(DURATION_SECONDS),
    "start_seconds": float(INPAINT_START_SECONDS),
    "end_seconds": float(INPAINT_END_SECONDS),
    "seed": int(INPAINT_SEED),
    "steps": 8,
    "cfg_scale": 1.0,
    "repo_revision": REVISION,
}
INPAINT_CONFIG_PATH = OUT_DIR / "inpainting_options.json"
INPAINT_CONFIG_PATH.write_text(
    json.dumps(INPAINT_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

INPAINT_SCRIPT = r"""
import hashlib, json, time
from pathlib import Path
import soundfile as sf
import torch
import torchaudio
from stable_audio_3 import StableAudioModel

config_path = Path("/content/stable_audio_music_pilot/inpainting_options.json")
config = json.loads(config_path.read_text(encoding="utf-8"))
source_path = Path(config["input_wav"])
output_path = Path(config["output_wav"])

waveform, sample_rate = torchaudio.load(str(source_path))
model = StableAudioModel.from_pretrained("small-music", device="cuda", model_half=True)

started = time.time()
audio = model.generate(
    inpaint_audio=(int(sample_rate), waveform),
    inpaint_mask_start_seconds=float(config["start_seconds"]),
    inpaint_mask_end_seconds=float(config["end_seconds"]),
    prompt=config["prompt"],
    duration=float(config["duration_seconds"]),
    steps=int(config["steps"]),
    cfg_scale=float(config["cfg_scale"]),
    seed=int(config["seed"]),
    batch_size=1,
)
result = audio[0].detach().float().cpu().numpy().T
output_path.parent.mkdir(parents=True, exist_ok=True)
sf.write(output_path, result, int(sample_rate), subtype="PCM_16")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "schema": "stable-audio-3-inpainting-pilot-v1",
    "model": "stabilityai/stable-audio-3-small-music",
    "repo_revision": config["repo_revision"],
    "source_file": source_path.name,
    "source_sha256": sha256_file(source_path),
    "output_file": output_path.name,
    "output_sha256": sha256_file(output_path),
    "sample_rate_hz": int(sample_rate),
    "duration_seconds": round(len(result) / int(sample_rate), 3),
    "inpaint_interval_seconds": [
        float(config["start_seconds"]),
        float(config["end_seconds"]),
    ],
    "prompt": config["prompt"],
    "seed": int(config["seed"]),
    "steps": int(config["steps"]),
    "cfg_scale": float(config["cfg_scale"]),
    "generation_seconds": round(time.time() - started, 2),
}
manifest_path = output_path.parent / "inpainting_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved:", output_path)
print("Duration:", manifest["duration_seconds"], "s | sample rate:", sample_rate, "Hz")
print("Generation time:", manifest["generation_seconds"], "s")
print("Manifest:", manifest_path)
"""

INPAINT_SCRIPT_PATH = Path("/content/run_stable_audio_inpainting.py")
INPAINT_SCRIPT_PATH.write_text(textwrap.dedent(INPAINT_SCRIPT), encoding="utf-8")
subprocess.run([str(PYTHON), str(INPAINT_SCRIPT_PATH)], check=True)


## Step 9 — compare before and after

Listen to the complete tracks first, then to the edited interval by itself. The full-track comparison tells us whether the edit belongs in the arrangement; the interval comparison makes the requested instrument easier to inspect.

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display, Markdown

base_audio, base_sr = sf.read(BASE_WAV, always_2d=True)
edited_audio, edited_sr = sf.read(INPAINT_WAV, always_2d=True)
if base_sr != edited_sr:
    raise RuntimeError(f"Sample-rate mismatch: {base_sr} vs {edited_sr}")

common_length = min(len(base_audio), len(edited_audio))
base_audio = base_audio[:common_length]
edited_audio = edited_audio[:common_length]
start_sample = int(INPAINT_START_SECONDS * base_sr)
end_sample = min(int(INPAINT_END_SECONDS * base_sr), common_length)

difference = edited_audio - base_audio
edit_mask = np.zeros(common_length, dtype=bool)
edit_mask[start_sample:end_sample] = True

def rms_value(values):
    return float(np.sqrt(np.mean(np.square(values)))) if values.size else 0.0

outside_delta_rms = rms_value(difference[~edit_mask])
inside_delta_rms = rms_value(difference[edit_mask])

COMPARISON = {
    "base_file": BASE_WAV.name,
    "edited_file": INPAINT_WAV.name,
    "base_seed": BASE_SEED,
    "inpaint_seed": INPAINT_SEED,
    "inpaint_interval_seconds": [INPAINT_START_SECONDS, INPAINT_END_SECONDS],
    "prompt": INPAINT_PROMPT,
    "outside_mask_delta_rms": outside_delta_rms,
    "inside_mask_delta_rms": inside_delta_rms,
    "human_review": {
        "harp_audible": None,
        "edit_blends_naturally": None,
        "outside_region_preserved": None,
        "overall_preference": None,
        "notes": ""
    }
}
COMPARISON_PATH = OUT_DIR / "inpainting_comparison.json"
COMPARISON_PATH.write_text(
    json.dumps(COMPARISON, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Outside edited interval — waveform delta RMS: {outside_delta_rms:.6f}")
print(f"Inside edited interval — waveform delta RMS: {inside_delta_rms:.6f}")
print("First listen for the music and transition; the RMS values are diagnostic only.")

display(Markdown(f"### Original — seed {BASE_SEED}"))
display(Audio(filename=str(BASE_WAV)))
display(Markdown(f"### Inpainted version — harp edit at {INPAINT_START_SECONDS:.0f}–{INPAINT_END_SECONDS:.0f} s"))
display(Audio(filename=str(INPAINT_WAV)))

display(Markdown("### Original segment"))
display(Audio(base_audio[start_sample:end_sample].T, rate=base_sr))
display(Markdown("### Edited segment"))
display(Audio(edited_audio[start_sample:end_sample].T, rate=edited_sr))

## Step 10 — download the paired comparison package

This ZIP contains the three original candidates, the selected base track, its inpainted full-mix version, generation manifests, and the saved experiment settings. It does not contain a separate harp stem, because this experiment tests inpainting rather than stem extraction.

In [ ]:
import zipfile
from google.colab import files

FINAL_ZIP = Path("/content/stable_audio_music_inpainting_pilot.zip")
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file() and path.suffix.lower() in {".wav", ".json"}:
            archive.write(path, arcname=path.name)

print("Paired comparison package:", FINAL_ZIP)
print("Included files:")
with zipfile.ZipFile(FINAL_ZIP, "r") as archive:
    for name in archive.namelist():
        print(" -", name)
files.download(str(FINAL_ZIP))